# 05 — Qwen3-1.7B en modo *think* con el pipeline del grafo

**Contexto:** el notebook 04 mostró que Qwen3-1.7B sin razonamiento (`/no_think`) es ~2.3× más
rápido que el 4B, pero con calidad comparable a la del 4B **v2** (sin grafo), no a la del 4B
**v3**. La pregunta de este notebook: ¿el razonamiento (`<think>`) le permite al 1.7B alcanzar
la calidad del 4B v3, y a qué costo de tiempo?

**Diferencias técnicas clave respecto al notebook 04:**

1. **Sin `/no_think`** — el modo razonamiento de Qwen3 queda activo (es el default del modelo).
2. **Sin gramática JSON** — `response_format` forzaría el JSON desde el primer token y el
   modelo nunca podría emitir el bloque `<think>`. Se le pide el JSON por instrucción, se
   parsea la salida, y la **validez del JSON pasa a ser una métrica del experimento** (en
   producción se recuperaría con un reintento o extrayendo con regex).
3. **`max_tokens=1500`** — el razonamiento consume tokens propios (200–1000 típicos) antes de
   la respuesta. Se mide cuántos gasta en pensar.
4. **`temperature=0.6, top_p=0.95`** — la recomendación oficial de Qwen para modo thinking
   (con temperatura alta el razonamiento divaga).

**Prerrequisitos:** notebooks 03 (grafo + `resultados_ab_grafo.csv`) y 04 (GGUF del 1.7B +
`resultados_qwen17b.csv`).

## 1. Carga del modelo y del grafo

In [1]:
import json
import re
import time
import unicodedata
from datetime import date
from pathlib import Path

import networkx as nx
import pandas as pd
from llama_cpp import Llama

MODEL_PATH = Path("../../models_registry/llm/Qwen3-1.7B-Q4_K_M.gguf").resolve()
assert MODEL_PATH.exists(), "Ejecuta primero el notebook 04 (descarga el GGUF del 1.7B)"

llm = Llama(
    model_path=str(MODEL_PATH),
    n_ctx=4096,
    n_threads=4,
    verbose=False,
)

with open("grafo_dominio.json") as f:
    G = nx.node_link_graph(json.load(f), edges="edges")

with open("drafts_ejemplo.json") as f:
    DRAFTS = [d["draft"] for d in json.load(f)]

AÑO_ACTUAL = date.today().year
print(f"Grafo: {G.number_of_nodes()} nodos | {len(DRAFTS)} drafts | año: {AÑO_ACTUAL}")

Grafo: 33 nodos | 7 drafts | año: 2026


## 2. Pipeline v3 (idéntico a los notebooks 03/04)

La única variable del experimento es el modo de razonamiento.

In [2]:
def _normalizar(texto: str) -> str:
    texto = unicodedata.normalize("NFKD", texto.lower().strip())
    return "".join(c for c in texto if not unicodedata.combining(c))

ALIAS_A_AMENIDAD = {
    _normalizar(alias): nodo
    for nodo, data in G.nodes(data=True) if data["tipo"] == "amenidad"
    for alias in data["aliases"] + [data["nombre"]]
}


def draft_json_a_texto(draft: dict) -> str:
    operacion = "RENTA" if draft["availableToRent"] else "VENTA"
    antiguedad = AÑO_ACTUAL - draft["constructionYear"]
    lineas = [
        f"Tipo de propiedad: {draft['propertyType']['name']}",
        f"Operación: {operacion}",
        f"Colonia: {draft['address']['neighborhoodName']}",
        f"Superficie: {draft['areaM2']:.0f} m²",
        f"Recámaras: {draft['bedrooms']}",
        f"Baños: {draft['bathrooms']}",
    ]
    if draft["parkingSpaces"] > 0:
        lineas.append(f"Estacionamientos: {draft['parkingSpaces']}")
    lineas.append(
        f"Año de construcción: {draft['constructionYear']}"
        + (f" ({antiguedad} años de antigüedad)" if antiguedad > 1 else " (a estrenar)")
    )
    lineas.append(f"En condominio: {'sí' if draft['condominium'] else 'no'}")
    if draft["amenities"]:
        lineas.append(f"Amenidades: {', '.join(draft['amenities'])}")
    return "\n".join(lineas)


def audiencias_para(draft: dict) -> list[str]:
    tipo = draft["propertyType"]["name"]
    rec, renta = draft["bedrooms"], draft["availableToRent"]
    if rec >= 4 or (rec >= 3 and tipo == "CASA"):
        return ["audiencia:familia_grande"]
    if rec >= 2:
        return ["audiencia:familia_pareja"]
    if renta:
        return ["audiencia:profesionista", "audiencia:persona_sola"]
    return ["audiencia:persona_sola"]


def contexto_desde_grafo(draft: dict) -> str:
    lineas = ["HECHOS Y ÁNGULOS APROBADOS (única fuente permitida además del draft):"]
    temas_usados: set[str] = set()

    op = "RENTA" if draft["availableToRent"] else "VENTA"
    lineas.append(f"- Operación {op}: {G.nodes[f'operacion:{op}']['tono']}.")
    nodo_tipo = f"tipo:{draft['propertyType']['name']}"
    if nodo_tipo in G:
        lineas.append(f"- Un(a) {draft['propertyType']['name'].lower()} {G.nodes[nodo_tipo]['narrativa']}.")
        temas_usados.update(G.successors(nodo_tipo))

    desconocidas = []
    for amenidad in draft["amenities"]:
        nodo = ALIAS_A_AMENIDAD.get(_normalizar(amenidad))
        if nodo is None:
            desconocidas.append(amenidad)
            continue
        for tema in G.successors(nodo):
            if tema in temas_usados:
                continue
            temas_usados.add(tema)
            frase = G.nodes[tema]["frases"][0]
            lineas.append(f"- {G.nodes[nodo]['nombre']} → {G.nodes[tema]['nombre']}: \"{frase}\".")
    if desconocidas:
        lineas.append(f"- Amenidades sin ángulo aprobado (menciónalas SOLO por su nombre): {', '.join(desconocidas)}.")

    frases_aud = [G.nodes[a]["frase"] for a in audiencias_para(draft)]
    lineas.append(f"- Audiencia sugerida: {'; '.join(frases_aud)}.")

    antiguedad = AÑO_ACTUAL - draft["constructionYear"]
    if antiguedad <= 1:
        lineas.append("- Antigüedad: a estrenar — puedes destacar que es de reciente construcción.")
    elif antiguedad <= 10:
        lineas.append("- Antigüedad: construcción reciente.")
    elif antiguedad > 30:
        lineas.append("- Antigüedad: más de 30 años — nómbrala como carácter e historia; "
                      "NO afirmes remodelaciones ni buen estado que el draft no indica.")
    if draft["areaM2"] >= 180:
        lineas.append("- Superficie: puedes describirla como amplia.")
    elif draft["areaM2"] <= 50:
        lineas.append("- Superficie: descríbela como compacta y eficiente, no como amplia.")
    return "\n".join(lineas)

## 3. Generación en modo *think*

Sin `/no_think` y sin gramática. La salida llega como `<think>razonamiento</think>` seguido del
JSON; se separan ambas partes y se extrae el JSON entre la primera `{` y la última `}`.

In [3]:
SYSTEM_PROMPT_V3 = """Eres un redactor inmobiliario profesional de México. Recibirás el DRAFT de una \
propiedad y un bloque de HECHOS Y ÁNGULOS APROBADOS. Escribirás el anuncio para un portal inmobiliario.

Reglas estrictas:
1. Usa ÚNICAMENTE la información del DRAFT y de los HECHOS Y ÁNGULOS APROBADOS. Puedes \
parafrasear las frases aprobadas con naturalidad, pero NO agregues características, lugares, \
vistas, cercanías ni cualidades que no aparezcan ahí.
2. PROHIBIDO mencionar precios, montos, rentas, mensualidades o cualquier cifra monetaria.
3. PROHIBIDO incluir datos de contacto, invitaciones a llamar, escribir o agendar visitas, \
y lenguaje de urgencia ("aprovecha", "no te lo pierdas", "últimos días", "oportunidad única").
4. La única ubicación que puedes mencionar es el nombre de la colonia, tal como aparece en el draft.
5. Usa español de México y vocabulario de la región: "recámaras" (nunca "habitaciones") y \
"estacionamientos" (nunca "cajones de estacionamiento" ni "plazas de garaje"). Si el draft no \
menciona estacionamientos, no hables de ellos.
6. El título debe tener máximo 10 palabras, atractivo sin ser sensacionalista.
7. La descripción debe tener entre 21 y 70 palabras, en párrafos fluidos (sin listas), \
con tono cálido y humano, sin mayúsculas sostenidas ni signos de admiración excesivos.

Responde exclusivamente con un JSON con las claves "titulo" y "descripcion"."""

RE_THINK = re.compile(r"<think>(.*?)</think>", re.S)


def generar_v3_think(draft: dict, temperature: float = 0.6, max_tokens: int = 1500) -> dict:
    mensaje = (f"DRAFT DE LA PROPIEDAD:\n{draft_json_a_texto(draft)}\n\n"
               f"{contexto_desde_grafo(draft)}")
    t0 = time.perf_counter()
    salida = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_V3},
            {"role": "user", "content": mensaje},
        ],
        temperature=temperature,
        top_p=0.95,
        max_tokens=max_tokens,
    )
    contenido = salida["choices"][0]["message"]["content"]
    tiempo_s = round(time.perf_counter() - t0, 1)

    m = RE_THINK.search(contenido)
    razonamiento = m.group(1).strip() if m else ""
    resto = contenido[m.end():] if m else contenido

    anuncio = {"titulo": None, "descripcion": None, "json_valido": False}
    inicio, fin = resto.find("{"), resto.rfind("}")
    if inicio != -1 and fin > inicio:
        try:
            parseado = json.loads(resto[inicio:fin + 1])
            anuncio = {"titulo": parseado.get("titulo"),
                       "descripcion": parseado.get("descripcion"),
                       "json_valido": True}
        except json.JSONDecodeError:
            pass

    anuncio.update(
        razonamiento=razonamiento,
        tokens_think=len(llm.tokenize(razonamiento.encode("utf-8"))) if razonamiento else 0,
        tokens_salida=salida["usage"]["completion_tokens"],
        tiempo_s=tiempo_s,
    )
    return anuncio


# Demo: el draft real, mostrando también CÓMO razonó
demo = generar_v3_think(DRAFTS[0])
print(f"⏱ {demo['tiempo_s']} s | think: {demo['tokens_think']} tok | salida total: {demo['tokens_salida']} tok\n")
print(f"RAZONAMIENTO (extracto):\n{demo['razonamiento'][:600]}...\n")
print(f"TÍTULO: {demo['titulo']}\n\n{demo['descripcion']}")

⏱ 67.0 s | think: 750 tok | salida total: 897 tok

RAZONAMIENTO (extracto):
Okay, I need to create an inmueble ad based on the provided draft and approved facts. Let me start by understanding the requirements.

First, the property is a casa in Prudencio Moscoso, 200 m², 4 bedrooms, 3 bathrooms, 2 parking spaces, built in 2025 (a estrenar). It's for rent, no condominium, and has a terrace, garden, and GYM. The key points are to avoid mentioning prices, contact info, urgency, and to use the exact colonia name. Also, use "recámaras" instead of "habitaciones" and "estacionamientos" instead of "cajones de estacionamiento" or "plazas de garaje".

The title must be 10 words ...

TÍTULO: Casa en Prudencio Moscoso

Ubicada en la colonia Prudencio Moscoso, esta casa de 200 m² ofrece espacios amplios para vivir. Con 4 recámaras, 3 baños y 2 estacionamientos, es ideal para familias. La terraza y el jardín permiten disfrutar del aire libre, mientras que el gimnasio facilita una rutina activa y salu

## 4. Corrida sobre los 7 drafts

Con el costo del razonamiento, espera ~30–60 s por anuncio (el experimento es justamente medir
si ese costo compra calidad de 4B).

In [4]:
RE_PRECIO = re.compile(r"\$|\bprecio\b|\bmxn\b|\bpesos?\b|mensualidad|mill[oó]n|\bmonto\b", re.I)
RE_CONTACTO = re.compile(
    r"cont[aá]ct|ll[aá]m[ae]|tel[eé]fono|whatsapp|escr[ií]b[ae]|agend[ae]|vis[ií]t[ae]|"
    r"aprovecha|no te lo pierdas|[uú]ltimos d[ií]as|cita|oportunidad [uú]nica", re.I)

filas = []
for draft in DRAFTS:
    anuncio = generar_v3_think(draft)
    texto = f"{anuncio['titulo'] or ''} {anuncio['descripcion'] or ''}"
    checks = {
        "menciona_precio": bool(RE_PRECIO.search(texto)),
        "menciona_contacto": bool(RE_CONTACTO.search(texto)),
        "n_palabras": len((anuncio["descripcion"] or "").split()),
    }
    filas.append({"draft_id": draft["id"], "variante": "v3-1.7B-think",
                  "titulo": anuncio["titulo"], "descripcion": anuncio["descripcion"],
                  "json_valido": anuncio["json_valido"], "tokens_think": anuncio["tokens_think"],
                  "tiempo_s": anuncio["tiempo_s"], **checks})
    marca = "⚠️" if (checks["menciona_precio"] or checks["menciona_contacto"]
                     or not anuncio["json_valido"]) else "✓"
    print(f"{marca} {draft['id']}: {anuncio['titulo']} "
          f"({anuncio['tiempo_s']} s, think {anuncio['tokens_think']} tok)")

df_think = pd.DataFrame(filas)
df_think.to_csv("resultados_qwen17b_think.csv", index=False)

print("\n--- Resumen Qwen3-1.7B think ---")
print(f"JSON válido: {int(df_think['json_valido'].sum())}/7 (sin gramática, era el riesgo)")
print(f"Violaciones precio/contacto: {int(df_think['menciona_precio'].sum() + df_think['menciona_contacto'].sum())}/7")
print(f"Tiempo promedio: {df_think['tiempo_s'].mean():.1f} s | think promedio: {df_think['tokens_think'].mean():.0f} tok")

✓ draft-001: Casa en Prudencio Moscoso: Vida Sana y Espacio para Crecer (73.8 s, think 1027 tok)
✓ draft-002: Departamento en El Cerrillo (66.6 s, think 905 tok)
✓ draft-003: Casa en Barrio de Guadalupe: Espacio Amplio y Histórico (42.9 s, think 513 tok)
✓ draft-004: Departamento en La Isla: Práctico y Líder (32.6 s, think 361 tok)
✓ draft-005: Casa en 31 de Marzo - Espacio para Familia (45.4 s, think 610 tok)
✓ draft-006: Departamento en Real del Monte (60.5 s, think 814 tok)
✓ draft-007: Loft en Centro: Espacio Práctico para Profesionales (35.2 s, think 423 tok)

--- Resumen Qwen3-1.7B think ---
JSON válido: 7/7 (sin gramática, era el riesgo)
Violaciones precio/contacto: 0/7
Tiempo promedio: 51.0 s | think promedio: 665 tok


## 5. Comparativa: las 4 variantes

La matriz completa del experimento — v2-4B y v3-4B (notebook 03), v3-1.7B sin think
(notebook 04) y v3-1.7B con think (este notebook).

In [ ]:
marcos = [df_think]
if Path("resultados_ab_grafo.csv").exists():
    df_4b = pd.read_csv("resultados_ab_grafo.csv")
    marcos.append(df_4b.assign(variante=df_4b["variante"].map({"v2": "v2-4B", "v3": "v3-4B"})))
if Path("resultados_qwen17b.csv").exists():
    marcos.append(pd.read_csv("resultados_qwen17b.csv"))

df_comp = pd.concat(marcos, ignore_index=True)

print("--- Resumen por variante ---")
display(df_comp.groupby("variante")[["menciona_precio", "menciona_contacto",
                                      "tiempo_s", "n_palabras"]].agg(
    {"menciona_precio": "sum", "menciona_contacto": "sum",
     "tiempo_s": "mean", "n_palabras": "mean"}).round(1))

# Lectura lado a lado del draft real del ejemplo
orden = ["v2-4B", "v3-4B", "v3-1.7B", "v3-1.7B-think"]
sub = df_comp[df_comp["draft_id"] == "draft-001"]
for variante in orden:
    fila = sub[sub["variante"] == variante]
    if fila.empty:
        continue
    fila = fila.iloc[0]
    print(f"\n{'=' * 70}\n### {fila['variante']} — {fila['titulo']} ({fila['tiempo_s']} s)\n")
    print(fila["descripcion"])

--- Resumen por variante ---


,menciona_precio,menciona_contacto,tiempo_s,n_palabras
variante,,,,
v2-4B,0,0,21.5,52.6
v3-1.7B,0,0,12.9,63.9
v3-1.7B-think,0,0,51.0,49.4
v3-4B,0,0,26.1,59.7



### v2-4B — Casa a estrenar en Prudencio Moscoso (31.2 s)

Linda casa de 200 m² a estrenar en la colonia Prudencio Moscoso, construida en 2025. Cuenta con 4 recámaras, 3 baños, estacionamientos para dos vehículos y espacios al aire libre como terraza y jardín. Disfruta de una vivienda moderna con acceso a un gimnasio. Ideal para quien busca comodidad y diseño actual en un entorno familiar.

### v3-4B — Casa de reciente construcción en Prudencio Moscoso (35.7 s)

Una casa amplia y moderna, a estrenar, en el corazón de Prudencio Moscoso. Ofrece espacio para vivir con tranquilidad y bienestar. Cuenta con terraza para disfrutar el aire libre sin salir de casa y un gimnasio para mantener una rutina activa y saludable. Ideal para familias que buscan un hogar con independencia y espacios para todos.

### v3-1.7B — Casa de 200 m² en Prudencio Moscoso (12.6 s)

La casa de 200 m² en Prudencio Moscoso ofrece espacio amplio para vivir en un ambiente acogedor. Con 4 recámaras, 3 baños y 2 estacion

: 

## 6. Evaluación

- [ ] **JSON válido:** ¿cuántos de 7 sin gramática? (si falla >1, el modo think necesita
      reintentos en producción — costo extra real)
- [ ] **Verificaciones automáticas:** 0 violaciones de precio/contacto/urgencia
- [ ] **Calidad vs v3-4B:** en la lectura lado a lado, ¿el think cierra la brecha?
- [ ] **Costo:** tiempo promedio vs los ~26 s del v3-4B — si piensa mucho, puede salir MÁS caro
      que el 4B directo
- [ ] **Razonamiento:** leer 1–2 trazas `<think>` — ¿razona sobre las reglas y el grafo, o
      divaga? (diagnóstico útil para afinar el prompt)

### Matriz de decisión final

| Ganador | Implicación para el worker |
|---|---|
| v3-4B | 4B + gramática JSON; ~50–60 s/anuncio en VPS; RAM ~4.5 GB |
| v3-1.7B (no think) | 1.7B + gramática JSON; ~20–25 s/anuncio; RAM ~2.3 GB; vigilar clichés |
| v3-1.7B-think | Solo si empata al 4B en calidad Y le gana en tiempo; requiere parser + reintentos de JSON |

Con el ganador definido: benchmark en el VPS y diseño del worker (`llama-server` + consumidor
de `llm_queue`).